<div style="text-align: center;" >
<h1 style="margin-top: 0.2em; margin-bottom: 0.1em;">Sentiment Analysis</h1>
<h4 style="margin-top: 0.7em; margin-bottom: 0.3em; font-style:italic">Ey you?! Yes, you! I am your mother!</h4>
</div>
<br>

You can find today's slides [here](https://docs.google.com/presentation/d/1Yt6xsRO2Svrlnua8yXjsh60Vj61mO2njW5n8zLjuFMs/edit?usp=sharing)

Today's topic is Sentiment Analysis. This is the approach of using natural language processing and text analysis to quantitatively analyze the sentiment of given documents. You can think of it as giving a piece of text or multiple texts to someone and asking them to tell you which emotions are conveyed in the given text(s) except that you are not giving the text(s) to a human but to a machine.<br>
At the basic level this analysis is only about detecting positive, negative, or neutral sentimental valence. An example for this rather simple method of sentiment analysis would be the VADER (Valence Aware Dictionary and sEntiment Reasoner) sentiment analyzer which will be introduced further down.<br>
Something a bit more advanced would be the identification of specific emotions like anger, hate, sadness, or joy. This can be done with LEIA (Linguistic Embeddings for the Identification of Affect).

Now, you might ask why sentiment analysis might be important. Simple example: Think of product reviews... Nobody wants to read hundreds, thousands of product reviews and keep track of whether people were sad, angry, joyful, or simply neutral about a product. Imagine all the hate you would be subjected to reading this stuff...<br>
So, the solution is pretty simple: Just have your machine do the 'reading'!<br>
Enough talk, let's get down to some analysis:

***
# VADER
## Setup

In [ ]:
import pandas as pd

# Sentiment analysis part
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

First we load in the sentiment analyzer. For this we have to download the specific analyzer we wish to use. In our case this is done with `nltk.download('vader_lexicon')`. You don't have to do this every time just once before using the analyzer for the first time. We then import the `SentimentIntensityAnalyzer` class which we will need to perform the actual sentiment analysis.<br>
If you have problems at any point and the explanation given here does not do it for you just check this [link](https://www.nltk.org/howto/sentiment.html) for a tutorial on how to use the nltk VADER sentiment analyzer.

In [ ]:
classifier = SentimentIntensityAnalyzer() # Create a classifier

We create an object of the class `SentimentIntesityAnalyzer`. This is used to perform the sentiment analysis on individual texts.

In [ ]:
data = ['I hate pineapple!',
        'I am not really sure what to make of our new prime minister',
        'Last night was so hilarious! I wish you would have been there with us...',
        'I do not hate homework',
        'I love my data',
        'Sometimes I feel down when I think about the state of our planet', 
        'When people say "Eat the rich" I am never quite certain whether they really mean to eat them...',
        "Seeing Aragorn completely murder Sauron's forces in The Lord of the Rings brought tears of joy to my face!"]

df = pd.DataFrame(data, columns=['text'])

In [ ]:
df

Next, we need to get our text data into a format that can be analyzed by the classifier. One option would be to create a list of separate texts and feed this to the classifier. Another option would be to input a column of a pandas data frame. Let's have a look at the output:

## Application

In [ ]:
for i in data: # using a list
    print(i, '\n', classifier.polarity_scores(i))

In [ ]:
for i in df['text']: # using a df column
    print(i, '\n', classifier.polarity_scores(i))

As you see the classifier assigned each of our example texts four different scores. A 'neg', 'neu', 'pos', and 'compound' score. You might already have guessed what the first three scores stand for. They hold the information about how negative, neutral, or positive a given text was classified to be. The fourth score, the compound score, is a combination of those three other scores. For general purposes you might consider a negative compound score to indicate an overall negative text, a positive compound score indicates a positive text, and a compound score of 0 indicates a neutral text. ***Note that this definition might vary depending on your specific use case.***

## Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

Ok, now that you have seen VADER in action let's talk about evaluating our classifications. Take some time and create a [heatmap](https://seaborn.pydata.org/generated/seaborn.heatmap.html) showing the real and predicted sentiments of our example texts. ***Hint: The cell above tells you which libraries you might want to use for the task. The cell below hints at what you might be missing as of now.***

In [ ]:
annotations = ['negative',
              'neutral',
              'positive',
              'neutral',
              'positive',
              'negative',
              'neutral',
              'positive']

df['annotation'] = annotations
df['prediction'] = None

df.head(1)

In [ ]:
comps = [] # empty list to store individual compound scores

for i in df['text']: # run VADER and extract specific compound scores
    comps.append(classifier.polarity_scores(i)['compound'])

df['compound'] = comps # create a column containing the compound scores

df.loc[df['compound'] < 0, 'prediction'] = 'negative' # depending on compound score assign label
df.loc[df['compound'] == 0, 'prediction'] = 'neutral'
df.loc[df['compound'] > 0, 'prediction'] = 'positive'

matrix = confusion_matrix(df['annotation'], df['prediction']) # create a confusion matrix of true/predicted labels

heatmap = sns.heatmap(matrix, annot = True,  cmap = "Greens") # plot the results
heatmap.set_title("Heatmap Showing Annotation Results\n");
heatmap.xaxis.set_ticklabels(['negative', 'neutral', 'positive'])
heatmap.yaxis.set_ticklabels(['negative', 'neutral', 'positive'])

One can see that the predictions and real labels did not match well. In a perfect scenario we would observe a deeply colored diagonal from the top left to the bottom right. This would indicate that predictions and true labels (nearly) always matched.<br>
The y-axis shows the true labels of the texts while the x-axis shows the predicted labels. Inspecting the above plot we see that the sentiment classification was not too successful in this example. ***Keep in mind though that we only had a sample of 8 different texts and that the 'true' labels were assigned by one human. Maybe different annotators would classify the texts differently...***

## Discussion

Now that we have applied the VADER sentiment analyzer to some texts let's think about possible strengths and weaknesses of our approach. Take a few minutes an try to come up with some pros and cons:

Cons:
- Can't extract subtle emotional valence
- Unaware of context ('I do not hate' -> negative)
- Gives only negative, neutral, or positive labels (in some cases e.g product reviews we already have this info (stars))

Pros:
- Easy to use
- Easy to understand
- (Can be) fast

***Note that this list is not exhaustive.***